# Flujo de entrenamiento - Regresión mensual

Este notebook sigue un flujo de entrenamiento, usando la vista `Fact_Dim.vw_modelo_predictivo_central_tarifa_mensual` como fuente principal y consolidando la predicción al nivel mensual.

In [1]:
import os
import sys
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import cross_val_score, train_test_split

ROOT_DIR = Path.cwd()
if not (ROOT_DIR / "src").exists():
    ROOT_DIR = ROOT_DIR.parent

SRC_DIR = ROOT_DIR / "src"
MODELOS_DIR = SRC_DIR / "modelos"
MODELOS_DIR.mkdir(parents=True, exist_ok=True)

if str(SRC_DIR) not in sys.path:
    sys.path.append(str(SRC_DIR))

os.environ.setdefault("PGDATABASE", "DW_Energia_ML")
os.environ.setdefault("PGUSER", "sa")
os.environ.setdefault("PGPASSWORD", "progra")
os.environ.setdefault("PGHOST", "localhost")
os.environ.setdefault("PGPORT", "5433")

from datos.CargadorDatos import CargadorDatos

## Cargar la vista modelo desde la BD

In [2]:
cargador = CargadorDatos()
cargador.sql_view_to_df("vw_modelo_predictivo_central_tarifa_mensual", schema="Fact_Dim")
df = cargador.df
df.head()

,anio,mes,trimestre,nombre_mes,nombre_empresa,sistema,tarifa,descripcion_tarifa,id_objecto,central_electrica,...,t2m_min,cloud_od,gwetroot,ts,prectotcorr,allsky_sfc_sw_dwn,ps,t2mwet,allsky_sfc_sw_diff,allsky_sfc_lw_dwn
0,2025,12,4,Diciembre,CNFL,DISTRIBUCION | GENERACION,ALUMBRADO PÚBLICO | COMERCIOS Y SERVICIOS | IN...,ALUMBRADO PÚBLICO | COMERCIOS Y SERVICIOS | IN...,8,P.E VALLE CENTRAL,...,17.800000,6.1738750000000000,0.850000,22.880000,2.550000,4.8871929824561404,93.710000,21.290000,2.0799629166666667,9.6989378289473684
1,2025,12,4,Diciembre,CNFL,DISTRIBUCION | GENERACION,ALUMBRADO PÚBLICO | COMERCIOS Y SERVICIOS | IN...,ALUMBRADO PÚBLICO | COMERCIOS Y SERVICIOS | IN...,12,VENTANAS,...,17.800000,6.1738750000000000,0.850000,22.880000,2.550000,4.8871929824561404,93.710000,21.290000,2.0799629166666667,9.6989378289473684
2,2025,12,4,Diciembre,CNFL,DISTRIBUCION | GENERACION,ALUMBRADO PÚBLICO | COMERCIOS Y SERVICIOS | IN...,ALUMBRADO PÚBLICO | COMERCIOS Y SERVICIOS | IN...,13,ANONOS,...,17.800000,6.1738750000000000,0.850000,22.880000,2.550000,4.8871929824561404,93.710000,21.290000,2.0799629166666667,9.6989378289473684
3,2025,12,4,Diciembre,CNFL,DISTRIBUCION | GENERACION,ALUMBRADO PÚBLICO | COMERCIOS Y SERVICIOS | IN...,ALUMBRADO PÚBLICO | COMERCIOS Y SERVICIOS | IN...,14,RÍO SEGUNDO,...,17.800000,6.1738750000000000,0.850000,22.880000,2.550000,4.8871929824561404,93.710000,21.290000,2.0799629166666667,9.6989378289473684
4,2025,12,4,Diciembre,CNFL,DISTRIBUCION | GENERACION,ALUMBRADO PÚBLICO | COMERCIOS Y SERVICIOS | IN...,ALUMBRADO PÚBLICO | COMERCIOS Y SERVICIOS | IN...,16,BRASIL,...,17.800000,6.1738750000000000,0.850000,22.880000,2.550000,4.8871929824561404,93.710000,21.290000,2.0799629166666667,9.6989378289473684


In [3]:
print(df.shape)
df.info()

(4608, 40)
<class 'pandas.DataFrame'>
RangeIndex: 4608 entries, 0 to 4607
Data columns (total 40 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   anio                          4608 non-null   str    
 1   mes                           4608 non-null   int64  
 2   trimestre                     4608 non-null   int64  
 3   nombre_mes                    4608 non-null   str    
 4   nombre_empresa                4608 non-null   str    
 5   sistema                       4608 non-null   str    
 6   tarifa                        4608 non-null   str    
 7   descripcion_tarifa            4608 non-null   str    
 8   id_objecto                    4608 non-null   int64  
 9   central_electrica             4608 non-null   str    
 10  operador                      4608 non-null   str    
 11  fuente                        4608 non-null   str    
 12  provincia                     4608 non-null   str    
 13  can

## Consolidar el dataset a grano mensual

La variable objetivo `pago_promedio_cliente_tarifa` es mensual, por eso el entrenamiento se hace con una sola fila por `empresa + año + mes`.

In [4]:
columnas_texto = [
    "nombre_mes",
    "sistema",
    "tarifa",
    "descripcion_tarifa",
    "provincia",
    "canton",
    "distrito",
]

columnas_sumar = [
    "abonados",
    "ventas_por_mes",
    "ingreso_sin_cvg",
    "ingreso_con_cvg",
]

columnas_promedio = [
    "precio_medio_sin_cvg",
    "precio_medio_con_cvg",
    "pago_promedio_cliente_tarifa",
    "coordenada_x",
    "coordenada_y",
    "t2m",
    "ws10m",
    "cloud_amt",
    "rh2m",
    "t2m_max",
    "t2m_min",
    "cloud_od",
    "gwetroot",
    "ts",
    "prectotcorr",
    "allsky_sfc_sw_dwn",
    "ps",
    "t2mwet",
    "allsky_sfc_sw_diff",
    "allsky_sfc_lw_dwn",
]

agregaciones = {col: "first" for col in columnas_texto}
agregaciones.update({col: "sum" for col in columnas_sumar})
agregaciones.update({col: "mean" for col in columnas_promedio})

df_modelo = (
    df.drop(columns=["id_objecto", "central_electrica", "operador", "fuente", "codigo_dta"], errors="ignore")
      .groupby(["nombre_empresa", "anio", "mes"], as_index=False)
      .agg(agregaciones)
)

print(df_modelo.shape)
df_modelo.head()

(672, 34)


,nombre_empresa,anio,mes,nombre_mes,sistema,tarifa,descripcion_tarifa,provincia,canton,distrito,...,t2m_min,cloud_od,gwetroot,ts,prectotcorr,allsky_sfc_sw_dwn,ps,t2mwet,allsky_sfc_sw_diff,allsky_sfc_lw_dwn
0,CNFL,2018,1,Enero,DISTRIBUCION | GENERACION,ALUMBRADO PÚBLICO | COMERCIOS Y SERVICIOS | IN...,ALUMBRADO PÚBLICO | COMERCIOS Y SERVICIOS | IN...,SAN JOSÉ,MORA,QUITIRRISÍ,...,17.816667,4.722222,0.741111,22.676667,3.836667,4.691444,94.623333,21.0,1.787578,9.627556
1,CNFL,2018,2,Febrero,DISTRIBUCION | GENERACION,ALUMBRADO PÚBLICO | COMERCIOS Y SERVICIOS | IN...,ALUMBRADO PÚBLICO | COMERCIOS Y SERVICIOS | IN...,SAN JOSÉ,MORA,QUITIRRISÍ,...,17.437778,4.15,0.615556,23.4,0.804444,6.258189,94.717778,20.603333,1.6309,9.383044
2,CNFL,2018,3,Marzo,DISTRIBUCION | GENERACION,ALUMBRADO PÚBLICO | COMERCIOS Y SERVICIOS | IN...,ALUMBRADO PÚBLICO | COMERCIOS Y SERVICIOS | IN...,SAN JOSÉ,MORA,QUITIRRISÍ,...,17.588889,4.306667,0.49,25.622222,0.883333,6.601011,94.611111,21.146667,1.816933,9.446
3,CNFL,2018,4,Abril,DISTRIBUCION | GENERACION,ALUMBRADO PÚBLICO | COMERCIOS Y SERVICIOS | IN...,ALUMBRADO PÚBLICO | COMERCIOS Y SERVICIOS | IN...,SAN JOSÉ,MORA,QUITIRRISÍ,...,19.39,5.272222,0.465556,26.445556,4.074444,5.508244,94.705556,22.243333,2.351189,9.848278
4,CNFL,2018,5,Mayo,DISTRIBUCION | GENERACION,ALUMBRADO PÚBLICO | COMERCIOS Y SERVICIOS | IN...,ALUMBRADO PÚBLICO | COMERCIOS Y SERVICIOS | IN...,SAN JOSÉ,MORA,QUITIRRISÍ,...,19.597778,6.294444,0.555556,25.023333,9.237778,5.006944,94.691111,22.57,2.445144,9.850133


## Preparar variables de entrada y salida

In [5]:
target = "pago_promedio_cliente_tarifa"
columnas_excluir = [
    "ingreso_sin_cvg",
    "ingreso_con_cvg",
    "precio_medio_sin_cvg",
    "precio_medio_con_cvg",
]

df_modelo = df_modelo.dropna(subset=[target]).copy()

X = df_modelo.drop(columns=[target, *columnas_excluir], errors="ignore")
X = pd.get_dummies(X, drop_first=False)
X = X.apply(pd.to_numeric, errors="coerce").fillna(0)

y = pd.to_numeric(df_modelo[target], errors="coerce")
mascara_valida = y.notna()
X = X.loc[mascara_valida].copy()
y = y.loc[mascara_valida].copy()

X.shape, y.shape

((672, 10161), (672,))

## Partición 70% entrenamiento, 15% validación, 15% test

In [6]:
X_train_val, X_test, y_train_val, y_test = train_test_split(
    X,
    y,
    test_size=0.15,
    random_state=42,
)

X_train, X_val, y_train, y_val = train_test_split(
    X_train_val,
    y_train_val,
    test_size=(15 / 85),
    random_state=42,
)

print("Train:", X_train.shape, y_train.shape)
print("Validation:", X_val.shape, y_val.shape)
print("Test:", X_test.shape, y_test.shape)

Train: (470, 10161) (470,)
Validation: (101, 10161) (101,)
Test: (101, 10161) (101,)


## Entrenar Regresión Lineal

In [7]:
modelo_lineal = LinearRegression()
cv_scores_lineal = cross_val_score(modelo_lineal, X_train, y_train, cv=5)
print(cv_scores_lineal)
print("Average 5-Fold CV Score:", np.mean(cv_scores_lineal))

modelo_lineal.fit(X_train, y_train)

[-3.04407466e+00 -1.22802295e+01 -1.30977052e-02 -2.00142764e+01
 -2.10760610e+01]
Average 5-Fold CV Score: -11.285547831901061


,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"tol tol: float, default=1e-6The precision of the solution (`coef_`) is determined by `tol` whichspecifies a different convergence criterion for the `lsqr` solver.`tol` is set as `atol` and `btol` of :func:`scipy.sparse.linalg.lsqr` whenfitting on sparse training data. This parameter has no effect when fittingon dense data... versionadded:: 1.7",1e-06
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation. This will only providespeedup in case of sufficiently large problems, that is if firstly`n_targets > 1` and secondly `X` is sparse or if `positive` is setto `True`. ``None`` means 1 unless in a:obj:`joblib.parallel_backend` context. ``-1`` means using allprocessors. See :term:`Glossary ` for more details.",None
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive. Thisoption is only supported for dense arrays.For a comparison between a linear regression model with positive constraintson the regression coefficients and a linear regression without such constraints,see :ref:`sphx_glr_auto_examples_linear_model_plot_nnls.py`... versionadded:: 0.24",False


## Entrenar Random Forest

In [8]:
modelo_rf = RandomForestRegressor(
    n_estimators=200,
    max_depth=12,
    random_state=42,
)

cv_scores_rf = cross_val_score(modelo_rf, X_train, y_train, cv=5)
print(cv_scores_rf)
print("Average 5-Fold CV Score:", np.mean(cv_scores_rf))

modelo_rf.fit(X_train, y_train)

[ 0.98105461  0.74148713 -0.01412427 -0.43893235  0.98603303]
Average 5-Fold CV Score: 0.4511036318858199


,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",200
,"criterion criterion: {""squared_error"", ""absolute_error"", ""friedman_mse"", ""poisson""}, default=""squared_error""The function to measure the quality of a split. Supported criteriaare ""squared_error"" for the mean squared error, which is equal tovariance reduction as feature selection criterion and minimizes the L2loss using the mean of each terminal node, ""friedman_mse"", which usesmean squared error with Friedman's improvement score for potentialsplits, ""absolute_error"" for the mean absolute error, which minimizesthe L1 loss using the median of each terminal node, and ""poisson"" whichuses reduction in Poisson deviance to find splits.Training using ""absolute_error"" is significantly slowerthan when using ""squared_error""... versionadded:: 0.18 Mean Absolute Error (MAE) criterion... versionadded:: 1.0 Poisson criterion.",'squared_error'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",12
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=1.0The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None or 1.0, then `max_features=n_features`... note:: The default of 1.0 is equivalent to bagged trees and more randomness can be achieved by setting smaller values, e.g. 0.3... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to 1.0.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",1.0
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples 

## Evaluación en validación y test

In [9]:
def evaluar_modelo(nombre, modelo, X_eval, y_eval):
    y_pred = modelo.predict(X_eval)
    mse = mean_squared_error(y_eval, y_pred)
    return {
        "modelo": nombre,
        "r2": modelo.score(X_eval, y_eval),
        "mse": mse,
        "rmse": np.sqrt(mse),
    }

metricas_validacion = pd.DataFrame([
    evaluar_modelo("linear_regression", modelo_lineal, X_val, y_val),
    evaluar_modelo("random_forest", modelo_rf, X_val, y_val),
]).set_index("modelo")

metricas_test = pd.DataFrame([
    evaluar_modelo("linear_regression", modelo_lineal, X_test, y_test),
    evaluar_modelo("random_forest", modelo_rf, X_test, y_test),
]).set_index("modelo")

metricas_validacion

,r2,mse,rmse
modelo,,,
linear_regression,-9.053879,1.360615e+09,36886.509072
random_forest,0.995799,5.685134e+05,753.998301


In [10]:
metricas_test

,r2,mse,rmse
modelo,,,
linear_regression,-7.250360,1.089419e+09,33006.341568
random_forest,-1.124116,2.804789e+08,16747.503374


## Guardar ambos modelos en `src/modelos`

In [11]:
columnas_modelo = X.columns.tolist()

ruta_lineal = MODELOS_DIR / "modelo_regresion_lineal_pago_promedio_cliente_tarifa.joblib"
ruta_rf = MODELOS_DIR / "modelo_random_forest_pago_promedio_cliente_tarifa.joblib"

joblib.dump(
    {
        "modelo": modelo_lineal,
        "columnas_modelo": columnas_modelo,
        "target": target,
    },
    ruta_lineal,
)

joblib.dump(
    {
        "modelo": modelo_rf,
        "columnas_modelo": columnas_modelo,
        "target": target,
    },
    ruta_rf,
)

print(ruta_lineal)
print(ruta_rf)

C:\Users\d3smo\Desktop\CUC\Progra 2\Proyectos\Proyecto_final_Colab\Prediccion_Consumo_Energia_Costa_Rica\src\modelos\modelo_regresion_lineal_pago_promedio_cliente_tarifa.joblib
C:\Users\d3smo\Desktop\CUC\Progra 2\Proyectos\Proyecto_final_Colab\Prediccion_Consumo_Energia_Costa_Rica\src\modelos\modelo_random_forest_pago_promedio_cliente_tarifa.joblib


## Recargar y usar ambos modelos

In [12]:
payload_lineal = joblib.load(ruta_lineal)
payload_rf = joblib.load(ruta_rf)

X_muestra = X_test.head().reindex(columns=payload_lineal["columnas_modelo"], fill_value=0)

pred_lineal = payload_lineal["modelo"].predict(X_muestra)
pred_rf = payload_rf["modelo"].predict(X_muestra)

pd.DataFrame({
    "real": y_test.head().values,
    "pred_lineal": pred_lineal,
    "pred_random_forest": pred_rf,
})

,real,pred_lineal,pred_random_forest
0,45309.015524,33603.250955,45309.147826
1,23795.177067,21425.403393,24255.774086
2,1455.693750,591.987043,1473.407271
3,17538.369417,15580.965984,17847.461413
4,17406.995560,29184.942081,17549.365452
